<a href="https://colab.research.google.com/github/Seth724/CNN-for-Image-Classification/blob/main/estimate%20MACs%20and%20Cpu%20latency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/pattern_cnn_experiments")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

print("Drive output folder:", DRIVE_ROOT)
print("Exists:", DRIVE_ROOT.exists())
print("Current contents:", list(DRIVE_ROOT.iterdir()))

Drive output folder: /content/drive/MyDrive/pattern_cnn_experiments
Exists: True
Current contents: [PosixPath('/content/drive/MyDrive/pattern_cnn_experiments/model_a_rmsprop_lr0.002_bs128_e20'), PosixPath('/content/drive/MyDrive/pattern_cnn_experiments/model_a_adam_lr0.003_bs64_e20'), PosixPath('/content/drive/MyDrive/pattern_cnn_experiments/model_a_comparison.csv'), PosixPath('/content/drive/MyDrive/pattern_cnn_experiments/model_a_report_artifacts.zip')]


In [3]:
%cd /content
!git clone https://github.com/Seth724/CNN-for-Image-Classification.git
%cd /content/CNN-for-Image-Classification

/content
Cloning into 'CNN-for-Image-Classification'...
remote: Enumerating objects: 441, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (212/212), done.
remote: Total 441 (delta 76), reused 207 (delta 50), pack-reused 171 (from 1)
Receiving objects: 100% (441/441), 34.95 MiB | 20.75 MiB/s, done.
Resolving deltas: 100% (139/139), done.
/content/CNN-for-Image-Classification


In [4]:
%cd /content/CNN-for-Image-Classification
!git pull

/content/CNN-for-Image-Classification
Already up to date.


In [5]:
!ls
!ls data/processed

data	   EfficientNetV2-B0  README.md		results  src
documents  MobileNetV2	      requirements.txt	scripts
fashion_mnist_70_15_15.npz  metadata.json  split_manifest.csv


In [6]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [7]:
!python scripts/verify_data.py

Verification PASSED
Dataset : Fashion-MNIST
Seed    : 42
Train   : (49000, 28, 28)
Val     : (10500, 28, 28)
Test    : (10500, 28, 28)


In [8]:
!grep -n "model_size_kb\|mean_epoch_time_s\|EpochTimer" scripts/train_model_a.py

17:from src.evaluation import EpochTimer, evaluate_classifier, plot_history
84:    epoch_timer = EpochTimer()
96:    model_size_kb = os.path.getsize(model_path) / 1024
129:            "model_size_kb": round(model_size_kb, 2),
130:            "model_size_mb": round(model_size_kb / 1024, 3),
131:            "mean_epoch_time_s": round(float(sum(steady_epoch_times) / len(steady_epoch_times)), 3),


In [25]:
!python scripts/train_model_a.py \
  --epochs 20 \
  --batch-size 128 \
  --optimizer rmsprop \
  --learning-rate 0.0003 \
  --seed 42 \
  --output-dir "/content/drive/MyDrive/pattern_cnn_experiments/model_a_rmsprop_lr0.002_bs128_e20"

W0000 00:00:1792344294.796340    3448 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
Model: "model_a"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_relu (Conv2D)             │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_relu (Conv2D)             │ (None, 14, 14, 64)     │        18,496 │
├───────────────────

In [26]:
# CPU latency and MAC measurement for the two final Model A runs

import json
from pathlib import Path

import tensorflow as tf
from tensorflow import keras

# This import should already work if the notebook previously imported src.evaluation.
from src.evaluation import estimate_flops, estimate_macs, measure_cpu_latency_ms


# Your existing Drive output folder
DRIVE_ROOT = Path("/content/drive/MyDrive/pattern_cnn_experiments")

# Correct final Model A runs
selected_runs = {
    "Model A - Adam": DRIVE_ROOT / "model_a_adam_lr0.003_bs64_e20",
    "Model A - RMSprop": DRIVE_ROOT / "model_a_rmsprop_lr0.002_bs128_e20",
}

results = []

for run_name, run_dir in selected_runs.items():
    # The training notebook saves the model inside the run directory.
    model_files = list(run_dir.rglob("*.keras"))

    if not model_files:
        raise FileNotFoundError(
            f"No .keras model found in {run_dir}. "
            "Check that the trained model was saved in this run directory."
        )

    model_path = model_files[0]
    print(f"Loading: {model_path}")

    # compile=False is correct because this is inference/profile measurement.
    model = keras.models.load_model(model_path, compile=False)

    # The helper measures one-image inference latency on CPU.
    cpu_latency_ms = measure_cpu_latency_ms(
        model,
        warmup_runs=10,
        measured_runs=100,
    )

    # estimate_macs internally estimates FLOPs and divides by two.
    flops = estimate_flops(model)
    macs = estimate_macs(model, flops=flops)

    result = {
        "run": run_name,
        "model_path": str(model_path),
        "total_parameters": int(model.count_params()),
        "flops_per_image": int(flops) if flops is not None else None,
        "macs_per_image": int(macs) if macs is not None else None,
        "macs_millions": round(macs / 1_000_000, 4) if macs is not None else None,
        "cpu_latency_median_ms": round(cpu_latency_ms, 4),
        "latency_warmup_runs": 10,
        "latency_measured_runs": 100,
    }

    results.append(result)

    print(f"CPU latency: {result['cpu_latency_median_ms']:.4f} ms")
    print(f"MACs: {result['macs_per_image']:,}")
    print(f"MACs: {result['macs_millions']:.4f} M")
    print("-" * 60)

# Display results
results_df = pd.DataFrame(results)
display(results_df)

# Save the new measurements
output_path = DRIVE_ROOT / "model_a_cpu_latency_macs.json"
with output_path.open("w") as file:
    json.dump(results, file, indent=2)

print(f"Saved measurements to: {output_path}")

Loading: /content/drive/MyDrive/pattern_cnn_experiments/model_a_adam_lr0.003_bs64_e20/models/model_a.keras


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


CPU latency: 1.3090 ms
MACs: 7,503,934
MACs: 7.5039 M
------------------------------------------------------------
Loading: /content/drive/MyDrive/pattern_cnn_experiments/model_a_rmsprop_lr0.002_bs128_e20/models/model_a.keras
CPU latency: 1.3054 ms
MACs: 7,503,934
MACs: 7.5039 M
------------------------------------------------------------


,run,model_path,total_parameters,flops_per_image,macs_per_image,macs_millions,cpu_latency_median_ms,latency_warmup_runs,latency_measured_runs
0,Model A - Adam,/content/drive/MyDrive/pattern_cnn_experiments...,101578,15007868,7503934,7.5039,1.3090,10,100
1,Model A - RMSprop,/content/drive/MyDrive/pattern_cnn_experiments...,101578,15007868,7503934,7.5039,1.3054,10,100


Saved measurements to: /content/drive/MyDrive/pattern_cnn_experiments/model_a_cpu_latency_macs.json


In [27]:
# Layer-wise analytical MAC calculation for Model A

import pandas as pd

layer_macs = [
    {
        "Layer": "Conv2D 1",
        "Output shape": "28 x 28 x 32",
        "MACs": 28 * 28 * 3 * 3 * 1 * 32,
    },
    {
        "Layer": "MaxPool 1",
        "Output shape": "14 x 14 x 32",
        "MACs": 0,
    },
    {
        "Layer": "Conv2D 2",
        "Output shape": "14 x 14 x 64",
        "MACs": 14 * 14 * 3 * 3 * 32 * 64,
    },
    {
        "Layer": "MaxPool 2",
        "Output shape": "7 x 7 x 64",
        "MACs": 0,
    },
    {
        "Layer": "Conv2D 3",
        "Output shape": "7 x 7 x 128",
        "MACs": 7 * 7 * 3 * 3 * 64 * 128,
    },
    {
        "Layer": "GlobalAveragePooling2D",
        "Output shape": "128",
        "MACs": 0,
    },
    {
        "Layer": "Dense 1",
        "Output shape": "64",
        "MACs": 128 * 64,
    },
    {
        "Layer": "Dense output",
        "Output shape": "10",
        "MACs": 64 * 10,
    },
]

macs_df = pd.DataFrame(layer_macs)

total_macs = int(macs_df["MACs"].sum())

macs_df["MACs"] = macs_df["MACs"].map(lambda value: f"{value:,}")

display(macs_df)

print(f"Total analytical MACs: {total_macs:,}")
print(f"Total analytical MACs: {total_macs / 1_000_000:.6f} M")

,Layer,Output shape,MACs
0,Conv2D 1,28 x 28 x 32,"225,792"
1,MaxPool 1,14 x 14 x 32,0
2,Conv2D 2,14 x 14 x 64,"3,612,672"
3,MaxPool 2,7 x 7 x 64,0
4,Conv2D 3,7 x 7 x 128,"3,612,672"
5,GlobalAveragePooling2D,128,0
6,Dense 1,64,"8,192"
7,Dense output,10,640


Total analytical MACs: 7,459,968
Total analytical MACs: 7.459968 M
